# 1. Amazon Bedrock Knowledge Bases - Creating RAG
- Tested in Amazon SageMaker AI - Notebook - JupyterLab environment.
- Kernel: conda_python3

Amazon Bedrock's Knowledge Bases allow you to securely connect Amazon Bedrock's Foundation models with your enterprise data to perform RAG. By accessing additional data, you can generate more relevant, contextual, and accurate responses without continuously retraining the model. Additionally, all information retrieved from the Knowledge Base includes source information, increasing transparency and minimizing model hallucinations.

![aos001](../img/aosrag-001.jpg)

## 1. Setup

### Installing Required Libraries

In [ ]:
# Install required libraries (kernel restart required after installation)
!pip install -q boto3 --upgrade
!pip install -q awscli --upgrade

### Importing Libraries and Setting Up Session

In [ ]:
# Import required libraries
import boto3
import json
import time
import os
import uuid
from datetime import datetime
import pandas as pd
from tqdm.notebook import tqdm

# Set AWS region
region = boto3.session.Session().region_name
print(f"Current AWS region: {region}")

# Set up session and clients
session = boto3.session.Session(region_name=region)
bedrock = session.client('bedrock')
bedrock_runtime = session.client('bedrock-runtime')
bedrock_agent = session.client('bedrock-agent')
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')
s3 = session.client('s3')

In [ ]:
print(boto3.__version__)

## 2. Creating KnowledgeBase - RAG 

<b> Select Knowledge Bases in the Amazon Bedrock console screen. </b>

![graphrag01](../img/graphrag-01.png)

<b> Click the Create button and select Knowledge Base with vector store. </b>

![aosrag02](../img/aosrag-02.jpg)

<b> Enter and select the following information: 
- Knowledge Base name: aosrag-workshop
- IAM permission: Select Create and use a new service role
- Choose data source: Select Amazon S3
Then click the Next button.
</b>

![aosrag03](../img/aosrag-03.jpg)

<b> Enter and select the following information: 
- Data source name: 10-q
- S3 URI: Enter or select the previously created S3 path s3://~/data/
Then click the Next button.
</b>

![aosrag04](../img/aosrag-04.jpg)

Select Titan Text Embeddings V2 as the embedding model.
![aosrag05](../img/aosrag-05.jpg)

Select OpenSearch Serverless as the Vector store, then click Next.
![aosrag06](../img/aosrag-06.jpg)

After reviewing the KB creation information, click the Create Knowledge Base button.
![aosrag07](../img/aosrag-07.jpg)

After a few minutes, the KB - OpenSearch Vector Store will be created, and you can check the following information:
<br>Copy this information for use in the next steps.
- Knowledge Base ID
![aosrag08](../img/aosrag-08.jpg)

Perform Data Sync in the Data source.
* Select the Data source (10-q).
* Click the Sync button and wait a few minutes for the Data Sync to complete. After completion, it will display Status: Available.
![aosrag09](../img/aosrag-09.jpg)

## 3. Running RAG

Save the Knowledge Base ID value you copied earlier to the variable below.

In [ ]:
kb_id = "Enter KB ID"

In [ ]:
def query_knowledge_base(query_text, model_id="anthropic.claude-3-5-sonnet-20241022-v2:0", max_tokens=1000):
    try:
        # Configure search request
        retrieve_response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=kb_id,
            retrievalQuery={
                'text': query_text
            },
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': 3,
                    'overrideSearchType': 'SEMANTIC'                    
                }
            }
        )
        
        # Check search results
        retrieved_results = retrieve_response.get('retrievalResults', [])
        
        if not retrieved_results:
            print("No search results found.")
            return None
        
        # Use search results as context
        context = ""
        for i, result in enumerate(retrieved_results):
            content = result['content']['text']
            source = result.get('location', {}).get('s3Location', {}).get('uri', 'Unknown source')
            score = result.get('score', 0)
            
            context += f"\n\nReference Document {i+1} (Relevance score: {score}):\n{content}\n"
            print(f"Search result {i+1}: Relevance score {score}")
        
        # Construct query for Claude
        prompt = f"""
Please answer the user's question. Use the information from the following reference documents:

{context}

User question: {query_text}

Answer:
"""
        
        # Send query to Claude (modified part)
        response = bedrock_runtime.converse(
            modelId=model_id,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
            inferenceConfig={
                "maxTokens": max_tokens,
                "temperature": 0.7,
                "topP": 0.9
            }
        )
        
        # Extract and return response
        answer = response['output']['message']['content'][0]['text']
        return answer
        
    except Exception as e:
        print(f"Error occurred during query execution: {e}")
        return None

In [ ]:
# Run example queries
test_queries = [
    "What was Amazon's total revenue and net income for Q1 2023?",
    "How did AWS (Amazon Web Services) performance in Q1 2023 compare to Q3 2023?",
    "What are the major changes in Amazon's operating cash flow and investment activities?"
]

for query in test_queries:
    print(f"\nQuestion: {query}")
    answer = query_knowledge_base(query)
    if answer:
        print(f"\nAnswer:\n{answer}")
    print("-" * 80)